In [1]:
!pip install sentence-transformers datasets huggingface_hub langchain-text-splitters pandas pyarrow

In [2]:
import json
import os
import random
from huggingface_hub import snapshot_download

print("Downloading dataset...")
dataset_path = snapshot_download(
    repo_id="vectara/open_ragbench",
    repo_type="dataset",
    local_dir="/tmp/open_ragbench"
)

corpus_path = os.path.join(dataset_path, "pdf", "arxiv", "corpus")
all_paper_files = os.listdir(corpus_path)

random.seed(42)
random.shuffle(all_paper_files)
paper_files = all_paper_files[:100]

print(f"Total papers available: {len(all_paper_files)}")
print(f"Papers selected: {len(paper_files)}")
print(f"First few selected: {paper_files[:3]}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1006 files:   0%|          | 0/1006 [00:00<?, ?it/s]

Total papers available: 1000
Papers selected: 100
First few selected: ['2501.00225v2.json', '2401.06959v2.json', '2412.16126v1.json']


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

documents = []  # for documents table
chunks = []     # for chunks table

for filename in paper_files:
    with open(os.path.join(corpus_path, filename)) as f:
        paper = json.load(f)

    source_id = paper["id"]
    title = paper.get("title", "")
    abstract = paper.get("abstract", "")
    authors = json.dumps(paper.get("authors", []))
    categories = json.dumps(paper.get("categories", []))
    published = paper.get("published", "")
    updated = paper.get("updated", "")

    documents.append({
        "source_id": source_id,
        "title": title,
        "abstract": abstract,
        "authors": authors,
        "categories": categories,
        "published": published,
        "updated": updated,
    })

    for section in paper.get("sections", []):
        text = section.get("text", "").strip()
        if not text:
            continue

        section_chunks = splitter.split_text(text)
        for chunk_text in section_chunks:
            chunks.append({
                "source_id": source_id,
                "section_id": str(section.get("section_id", "")),
                "content": chunk_text,
            })

print(f"Documents: {len(documents)}")
print(f"Total chunks: {len(chunks)}")
print(f"Average chunks per paper: {len(chunks) / len(documents):.1f}")

Documents: 100
Total chunks: 24846
Average chunks per paper: 248.5


In [4]:
import time
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Model device: {model.device}")

texts = [c["content"] for c in chunks]

start = time.time()
embeddings = model.encode(texts, batch_size=128, show_progress_bar=True)
elapsed = time.time() - start

print(f"\nEmbedded {len(embeddings)} chunks in {elapsed:.1f}s")
print(f"Embedding dimension: {embeddings.shape[1]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model device: cuda:0


Batches:   0%|          | 0/195 [00:00<?, ?it/s]


Embedded 24846 chunks in 55.3s
Embedding dimension: 384


In [5]:
import pandas as pd

df_documents = pd.DataFrame(documents)

df_chunks = pd.DataFrame(chunks)
df_chunks["embedding"] = [e.tolist() for e in embeddings]

print("Documents shape:", df_documents.shape)
print("Chunks shape:", df_chunks.shape)
print("\nDocuments sample:")
print(df_documents.head(2))
print("\nChunks sample (without embedding):")
print(df_chunks[["source_id", "section_id", "content"]].head(2))

df_documents.to_parquet("/tmp/documents.parquet", index=False)
df_chunks.to_parquet("/tmp/chunks.parquet", index=False)

docs_size = os.path.getsize("/tmp/documents.parquet") / (1024 * 1024)
chunks_size = os.path.getsize("/tmp/chunks.parquet") / (1024 * 1024)
print(f"\ndocuments.parquet size: {docs_size:.1f} MB")
print(f"chunks.parquet size: {chunks_size:.1f} MB")
print(f"Total size: {docs_size + chunks_size:.1f} MB")

Documents shape: (100, 7)
Chunks shape: (24846, 4)

Documents sample:
      source_id                                              title  \
0  2501.00225v2  Complexified tetrahedrons, fundamental groups,...   
1  2401.06959v2  Quantifying energy landscape of high-dimension...   

                                            abstract  \
0  In this paper, the volume conjecture for doubl...   
1  High-dimensional networks producing oscillator...   

                                             authors  \
0                                   ["Jun Murakami"]   
1  ["Shirui Bian", "Ruisong Zhou", "Wei Lin", "Ch...   

                 categories             published               updated  
0               ["math.GT"]  2024-12-31T02:21:17Z  2025-03-04T07:07:42Z  
1  ["q-bio.QM", "q-bio.MN"]  2024-01-13T03:15:13Z  2025-01-31T10:38:07Z  

Chunks sample (without embedding):
      source_id section_id                                            content
0  2501.00225v2          0                   

In [6]:
df_chunks.to_parquet("/tmp/chunks_compressed.parquet", index=False, compression="gzip")

compressed_size = os.path.getsize("/tmp/chunks_compressed.parquet") / (1024 * 1024)
print(f"Compressed chunks.parquet size: {compressed_size:.1f} MB")

Compressed chunks.parquet size: 42.6 MB


In [7]:
from google.colab import files
files.download("/tmp/documents.parquet")
files.download("/tmp/chunks.parquet")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>